In [ ]:
# making sure we only get take in 2024 data
import pandas as pd



raw_deliveries = pd.read_excel('raw_data/deliveries.xlsx')
season2024_deliveries = raw_deliveries[raw_deliveries['match_id'].isin(season2024_matches['id'])]


#print(raw_matches.head())
print(season2024_deliveries.head())



In [ ]:
print(raw_matches.dtypes)

In [36]:

season2024_deliveries.to_excel('season2024_deliveries.xlsx', index=False)


# Note: Three matches are missing from the matches file and the corresponding deliveries are missing from the deliveries file because
# the matches were washed out and no deliveries were bowled 


In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_excel('2024_raw_data/season2024_deliveries.xlsx')

# Concatenate player names from the three columns
all_names = pd.concat([df['bowler'], df['batter'], df['non_striker']])

# Remove missing values and get unique names
unique_names = sorted(all_names.dropna().unique())

# Print the list of unique player names
print(unique_names)
print(len(unique_names))

In [ ]:
import time
from googlesearch import search
import requests
from urllib.parse import quote

# Your list of names
names = unique_names

def get_first_cricbuzz_link(name):
    """
    Search for 'cricbuzz name' and return the first link
    """
    try:
        # Create search query
        query = f"cricbuzz {name}"
        print(f"Searching for: {query}")
        
        # Perform Google search and get first result
        # num=1 means we only want 1 result
        # stop=1 means stop after 1 result
        search_results = search(query)
        
        # Get the first (and only) result
        first_link = next(search_results, None)
        
        if first_link:
            print(f"Found: {first_link}")
            return first_link
        else:
            print(f"No results found for {name}")
            return None
            
    except Exception as e:
        print(f"Error searching for {name}: {str(e)}")
        return None

def main():
    results = {}
    
    for i, name in enumerate(names, 1):
        print(f"\n--- Processing {i}/{len(names)}: {name} ---")
        
        link = get_first_cricbuzz_link(name)
        results[name] = link
        
        # Add delay to avoid being blocked by Google
        # Increase this if you get blocked
        time.sleep(5)
    
    # Print all results
    print("\n" + "="*50)
    print("FINAL RESULTS:")
    print("="*50)
    
    for name, link in results.items():
        if link:
            print(f"{name}: {link}")
        else:
            print(f"{name}: NO RESULT FOUND")
    
    # Optionally save to file
    with open("cricbuzz_links.txt", "w") as f:
        for name, link in results.items():
            f.write(f"{name}: {link if link else 'NO RESULT'}\n")
    
    print(f"\nResults saved to cricbuzz_links.txt")

if __name__ == "__main__":
    main()

In [ ]:
import csv

def extract_cricbuzz_links():
    # Read the input file
    with open('cricbuzz_links.txt', 'r', encoding='utf-8') as file:
        content = file.read()
    
    # Split into lines and extract valid URLs
    lines = content.strip().split('\n')
    valid_links = []
    
    for line in lines:
        line = line.strip()
        if line and ': ' in line:
            parts = line.split(': ', 1)  # Split only on first occurrence
            if len(parts) == 2:
                player_name = parts[0].strip()
                url = parts[1].strip()
                
                # Only include valid HTTP URLs (filter out /search?num=12 type entries)
                if url.startswith('http'):
                    valid_links.append([player_name, url])
    
    # Write to CSV file
    with open('cricbuzz_links.csv', 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['Player Name', 'URL'])  # Header
        writer.writerows(valid_links)
    
    print(f"✅ Successfully extracted {len(valid_links)} valid links")
    print("📁 Saved to: cricbuzz_links.csv")
    
    # Show first few entries as preview
    print("\n📋 Preview of extracted data:")
    print("-" * 50)
    for i, (name, url) in enumerate(valid_links[:5]):
        print(f"{name}: {url}")
    if len(valid_links) > 5:
        print(f"... and {len(valid_links) - 5} more entries")

if __name__ == "__main__":
    try:
        extract_cricbuzz_links()
    except FileNotFoundError:
        print("❌ Error: cricbuzz_links.txt file not found!")
        print("Make sure the file is in the same directory as this script.")
    except Exception as e:
        print(f"❌ Error: {str(e)}")

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

dic={"Name": [], "Photo_url": [], "Born": [], "Birth_Place": [], "Height": [], "Role": [], "Batting_Style": [], "Bowling_Style": []}

def get_player_info(profile_url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    
    res = requests.get(profile_url, headers=headers)
    if res.status_code != 200:
        return None
    
    soup = BeautifulSoup(res.text, "html.parser")
    
    # Extract player name
    name_tag = soup.find("h1", class_="cb-font-40")
    name = name_tag.text.strip() if name_tag else None
    
    # Extract player photo URL (get the second image which is the player photo)
    img_tags = soup.find_all("img")
    photo_url = None
    if len(img_tags) >= 2:
        photo_url = img_tags[1].get("src")  # Second image is the player photo
    
    # Extract personal information
    info_dict = {}
    cricket_info_terms = {
        "Born": ["Born"],
        "Birth Place": ["Birth Place"],
        "Height": ["Height"],
        "Role": ["Role"],
        "Batting Style": ["Batting Style"],
        "Bowling Style": ["Bowling Style"]
    }
    
    for key, terms in cricket_info_terms.items():
        for term in terms:
            elements = soup.find_all(string=re.compile(f"\\b{term}\\b", re.IGNORECASE))
            for element in elements:
                parent = element.parent
                if parent and 'cb-col-40' in parent.get('class', []):
                    next_sibling = parent.find_next_sibling()
                    if next_sibling:
                        value = next_sibling.text.strip()
                        if value and len(value) < 100:
                            info_dict[key] = value
                            break
            if key in info_dict:
                break
    
    dic["Name"].append(name)
    dic["Photo_url"].append(photo_url)
    dic["Born"].append(info_dict.get("Born", ""))
    dic["Birth_Place"].append(info_dict.get("Birth Place", ""))
    dic["Height"].append(info_dict.get("Height", ""))
    dic["Role"].append(info_dict.get("Role", ""))
    dic["Batting_Style"].append(info_dict.get("Batting Style", ""))
    dic["Bowling_Style"].append(info_dict.get("Bowling Style", ""))
    
    return {
        "Name": name,
        "Photo_url": photo_url,
        "Born": info_dict.get("Born", ""),
        "Birth_Place": info_dict.get("Birth Place", ""),
        "Height": info_dict.get("Height", ""),
        "Role": info_dict.get("Role", ""),
        "Batting_Style": info_dict.get("Batting Style", ""),
        "Bowling_Style": info_dict.get("Bowling Style", "")
    }

# Example usage
linkscsv = pd.read_csv('cricbuzz_links.csv')
links = linkscsv['URL'].tolist()

for link in links:
    player_info = get_player_info(link)
    if player_info:
        print(f"Processed: {player_info['Name']}")
    else:
        print(f"Failed to process link: {link}")


# Save the collected data to a DataFrame and then to an Excel file
player_df = pd.DataFrame(dic)
player_df.to_excel('cricbuzz_player_info.xlsx', index=False)



In [ ]:
import pandas as pd
playerdf = pd.read_excel('2024_raw_data/2024_player_info.xlsx')
print(playerdf.head(),end='\n\n')
print(playerdf.dtypes, end='\n\n')
print(playerdf.info(), end='\n\n')

deliveriesdf = pd.read_excel('2024_raw_data/season2024_deliveries.xlsx')
print(deliveriesdf.head(), end='\n\n')
print(deliveriesdf.dtypes, end='\n\n')
print(deliveriesdf.info(), end='\n\n')

matchesdf = pd.read_excel('2024_raw_data/season2024_matches.xlsx')
print(matchesdf.head(), end='\n\n')
print(matchesdf.dtypes, end='\n\n')
print(matchesdf.info(), end='\n\n')


In [ ]:
# creation of the batting summary dataframe and generation of the .xlsx file

batting_summary = pd.DataFrame(columns=['Player','Innings','Not outs', 'Runs', 'Average', 'Strike Rate', '100s', '50s','4s', '6s'])
batting_summary['Player'] = deliveriesdf['batter'].unique()
print(batting_summary.head(), end='\n\n')
# Calculate runs played by each player
runs_by_batter = deliveriesdf.groupby('batter')['batsman_runs'].sum()
batting_summary['Runs'] = batting_summary['Player'].map(runs_by_batter)
print(batting_summary.head(), end='\n\n')
# Calculate innings played by each player
innings_by_batter = deliveriesdf.groupby('batter')['match_id'].nunique()
batting_summary['Innings'] = batting_summary['Player'].map(innings_by_batter)
print(batting_summary.head(), end='\n\n')
# Testing a possible issue with the innings count
# print(deliveriesdf[deliveriesdf['batter'] == 'V Kohli']['match_id'].nunique())
unique_outs = deliveriesdf[deliveriesdf['is_wicket'] == 1][['match_id', 'player_dismissed']].drop_duplicates()
# Now count the number of unique matches where each batter got out
outs_by_batter = unique_outs.groupby('player_dismissed')['match_id'].count()
# next query counts the number of outs for each player by subtracting the number of outs from number of innings while having player mapped throughout
batting_summary['Not outs'] = batting_summary['Player'].map(innings_by_batter) - batting_summary['Player'].map(outs_by_batter)
'''
Testing the code
print(outs_by_batter.head(), end='\n\n')
print(outs_by_batter.dtypes, end='\n\n')
print(outs_by_batter['V Kohli'])
'''
'''
There was an issue with RM Patidar's outs count, it shows 14 outs while he has only played 13 innings. This is because
in one of the match with the id 1426306, its been registered that he got out twice in the same innings,
he was given under the 'batter' column even when another player Faf du Plessis was the one who got out effectively
giving him an extra out in the same match.
Realised it was the same for Rahane as well, so fixed it.
But this is a broad issue which can be fixed by chcecking the 'is_wicket' column with unique 'match_id' values


Another hindsight from me is the fact that we need to take into account'the player dismissed' column because there are run outs :cries:
'''
#print(batting_summary[batting_summary['Not outs'] < 0][['Player', 'Not outs']], end='\n\n')
print(batting_summary.head(), end='\n\n')

# Calculate average
batting_summary['Average'] = batting_summary['Runs'] / (batting_summary['Innings'] - batting_summary['Not outs'])

print(batting_summary.head(), end='\n\n')


# Calculate strike rate
balls_per_bastman =  deliveriesdf[deliveriesdf['extras_type']!='wides'].groupby('batter')['ball'].count() 

'''
Strike rate was calculated wrong in the beginning because wide balls were not taken into account as
the batter name was still given and it was being considered as a ball faced by the batter
now its fixed by excluding the wide balls from the count
'''

batting_summary['Strike Rate'] = batting_summary["Runs"] / batting_summary['Player'].map(balls_per_bastman) * 100


print(batting_summary.head(), end='\n\n')

# Calculate 4s nd 6s

# Count of sixes per batter
sixes = deliveriesdf[deliveriesdf['batsman_runs'] == 6].groupby('batter').size()

# Count of fours per batter
fours = deliveriesdf[deliveriesdf['batsman_runs'] == 4].groupby('batter').size()

# Map to batting_summary
batting_summary['6s'] = batting_summary['Player'].map(sixes)
batting_summary['4s'] = batting_summary['Player'].map(fours)

# Fill NaN with 0 for batters who scored no 4s or 6s
batting_summary['6s'] = batting_summary['6s'].fillna(0).astype(int)
batting_summary['4s'] = batting_summary['4s'].fillna(0).astype(int)

print(batting_summary.head(), end='\n\n')

# Calculate 100s and 50s

batsman_runs_per_match = deliveriesdf.groupby(['match_id', 'batter'])['batsman_runs'].sum().reset_index()

print(batsman_runs_per_match.head(), end='\n\n')

centuries = batsman_runs_per_match[batsman_runs_per_match['batsman_runs'] >= 100].groupby('batter').size()
fifties = batsman_runs_per_match[(batsman_runs_per_match['batsman_runs'] >= 50) & (batsman_runs_per_match['batsman_runs'] < 100)].groupby('batter').size()

batting_summary['100s'] = batting_summary['Player'].map(centuries).fillna(0).astype(int)
batting_summary['50s'] = batting_summary['Player'].map(fifties).fillna(0).astype(int)

print(batting_summary.head(), end='\n\n')


# Save the batting summary to an Excel file

batting_summary.to_excel('2024_raw_data/batting_summary.xlsx', index=False)



In [ ]:
print(balls_per_bastman)

In [ ]:
# creation of the bowling summary dataframe and generation of the .xlsx file
import pandas as pd

deliveriesdf = pd.read_excel('2024_raw_data/season2024_deliveries.xlsx')

bowling_summary = pd.DataFrame(columns=['Player','Innings Bowled','Runs Conceded','Economy', 'Average', 'Strike Rate', 'Wickets', '4 Wickets', '5 Wickets'])
bowling_summary['Player'] = deliveriesdf['bowler'].unique()
print(bowling_summary.head(), end='\n\n')
# Calculate innings bowled by by each bowler

innings_by_bowler = deliveriesdf.groupby('bowler')['match_id'].nunique()
bowling_summary['Innings Bowled'] = bowling_summary['Player'].map(innings_by_bowler)
print(bowling_summary.head(), end='\n\n')

# Calculate runs conceded by each bowler proceeded by economy
# Sum batsman_runs for all deliveries
batsman_runs_by_bowler = deliveriesdf.groupby('bowler')['batsman_runs'].sum()

# Sum extra_runs only when not byes/legbyes
filtered_extras = deliveriesdf[~deliveriesdf['extras_type'].isin(['byes', 'legbyes',])]
extra_runs_by_bowler = filtered_extras.groupby('bowler')['extra_runs'].sum()

# Combine them
runs_by_bowler = batsman_runs_by_bowler.add(extra_runs_by_bowler, fill_value=0)

bowling_summary['Runs Conceded'] = bowling_summary['Player'].map(runs_by_bowler)

'''
To calculate the economy, we also need to calculate the number of balls bowled by each bowler,
for this we do not consider wides and no balls and only consider the rest of the deliveries as legal
deliveries. 
'''

filtered_extras_for_balls_bowled = deliveriesdf[~deliveriesdf['extras_type'].isin(['wides', 'noballs'])]
balls_bowled_by_bowler = filtered_extras_for_balls_bowled.groupby('bowler')['ball'].count()

economy_per_bowler = (runs_by_bowler * 6) / balls_bowled_by_bowler

bowling_summary['Economy'] = bowling_summary['Player'].map(economy_per_bowler)

print(bowling_summary.head(), end='\n\n')

# Calculating average 
# First calculate wickets taken for this(dont take run outs,obstructing for bowler wickets during a dismissal)

filtered_df_for_wickets_taken = deliveriesdf[~deliveriesdf['dismissal_kind'].isin(['run out', 'obstructing the field'])]
wickets_taken_by_bowler = filtered_df_for_wickets_taken[filtered_df_for_wickets_taken['is_wicket'] == 1].groupby('bowler')['is_wicket'].count()

bowling_summary['Wickets'] = bowling_summary['Player'].map(wickets_taken_by_bowler)

print(bowling_summary.head(), end='\n\n')

# Calculate average by dividing runs conceded by wickets taken

bowling_summary['Average'] = bowling_summary['Runs Conceded'] / bowling_summary['Wickets']

print(bowling_summary.head(), end='\n\n')

# Calculate strike rate by diving balls bowled by wickets taken

strike_rate_bowler = balls_bowled_by_bowler / wickets_taken_by_bowler

bowling_summary['Strike Rate'] = bowling_summary['Player'].map(strike_rate_bowler)


print(bowling_summary.head(), end='\n\n')



'''
To calculate 4 and 5 wicket hauls, first calculate wickets taken per match and also by bowler
so use double group by on bowler and match_id
'''


wickets_taken_by_bowler_per_match = (
    filtered_df_for_wickets_taken[filtered_df_for_wickets_taken['is_wicket'] == 1]
    .groupby(['bowler', 'match_id'])['is_wicket']
    .count()
)


'''
To calculate 4 wicket hauls
first we filter the multindexed wickets_taken_by_bowler_per match so that we have only those matches with 4 or more wickets
Then we reset the index to convert it into a regular data frame
Then we can group by the bowler
size() gives the number of 4 wicket hauls
'''

four_wicket_hauls_per_bowler = (
    wickets_taken_by_bowler_per_match[wickets_taken_by_bowler_per_match == 4]
    .reset_index()
    .groupby('bowler')
    .size()
)


print(four_wicket_hauls_per_bowler.head(), end='\n\n')

bowling_summary['4 Wickets'] = bowling_summary['Player'].map(four_wicket_hauls_per_bowler)

'''
Follow the same procedure for
5 wicket hauls
'''

five_wicket_hauls_per_bowler = (
    wickets_taken_by_bowler_per_match[wickets_taken_by_bowler_per_match == 5]
    .reset_index()
    .groupby('bowler')
    .size()
)


print(five_wicket_hauls_per_bowler.head(), end='\n\n')

bowling_summary['5 Wickets'] = bowling_summary['Player'].map(five_wicket_hauls_per_bowler)


'''

Make sure you fill the NaN's with 0 where a bowler has
no 4 wicket or 5 wicket hauls
'''

bowling_summary.fillna(0, inplace=True)

print(bowling_summary.head(),end='\n\n')

bowling_summary.to_excel('2024_raw_data/bowling_summary.xlsx', index=False)


In [1]:
'''
getting csv files
from the excel files
'''

import pandas as pd

df1 = pd.read_excel('2024_raw_data.xlsx//batting_summary.xlsx')
df2 = pd.read_excel('2024_raw_data.xlsx//bowling_summary.xlsx')
df3 = pd.read_excel('2024_raw_data.xlsx//season2024_matches.xlsx')
df4 = pd.read_excel('2024_raw_data.xlsx//2024_player_info.xlsx')

# Save DataFrames to CSV files

df1.to_csv('2024_raw_data.csv//batting_summary.csv', index=False)
df2.to_csv('2024_raw_data.csv//bowling_summary.csv', index=False)
df3.to_csv('2024_raw_data.csv//season2024_matches.csv', index=False)
df4.to_csv('2024_raw_data.csv//2024_player_info.csv', index=False)

In [ ]:
'''
Fixing the names in kaggle related sources
'''

'''
First we get the names from the player_info file and then take only the last word in the name column
'''

import pandas as pd

# Load player info and check columns
player_info_df = pd.read_csv('2024_raw_data.csv/2024_player_info.csv')
print(player_info_df.columns)  # Check actual column names

# Standardize column names (remove spaces, lower case)
player_info_df.columns = player_info_df.columns.str.strip().str.lower()
print(player_info_df.columns)

# Now use the correct column name, e.g., 'name'
# Build a mapping from last name to full name
player_info_df['last_name'] = player_info_df['name'].str.split().str[-1]
last_name_to_full_name = player_info_df.set_index('last_name')['name'].to_dict()

# Load batting summary
batting_summary_df = pd.read_csv('2024_raw_data.csv/batting_summary.csv')
batting_summary_df['Player_full'] = batting_summary_df['Player'].apply(
    lambda x: last_name_to_full_name.get(x.split()[-1], x)
)

print(batting_summary_df[['Player', 'Player_full']].head())

# Remove the old Player column and rename Player_full to Player
batting_summary_df = batting_summary_df.drop(columns=['Player'])
batting_summary_df = batting_summary_df.rename(columns={'Player_full': 'Player'})

# Save the updated DataFrame back to CSV
batting_summary_df.to_csv('2024_raw_data.csv/batting_summary.csv', index=False)

print(batting_summary_df.head())

In [ ]:
# --- Fixing the names in bowling_summary using player_info ---
import pandas as pd

# Load player info and standardize column names
player_info_df = pd.read_csv('2024_raw_data.csv/2024_player_info.csv')
player_info_df.columns = player_info_df.columns.str.strip().str.lower()
player_info_df['last_name'] = player_info_df['name'].str.split().str[-1]
last_name_to_full_name = player_info_df.set_index('last_name')['name'].to_dict()

# Load bowling summary
bowling_summary_df = pd.read_csv('2024_raw_data.csv/bowling_summary.csv')
bowling_summary_df['Player_full'] = bowling_summary_df['Player'].apply(
    lambda x: last_name_to_full_name.get(x.split()[-1], x)
)

print(bowling_summary_df[['Player', 'Player_full']].head())

# Remove the old Player column and rename Player_full to Player
bowling_summary_df = bowling_summary_df.drop(columns=['Player'])
bowling_summary_df = bowling_summary_df.rename(columns={'Player_full': 'Player'})

# Save the updated DataFrame back to CSV
bowling_summary_df.to_csv('2024_raw_data.csv/bowling_summary.csv', index=False)

print(bowling_summary_df.head())

In [ ]:

# Create a mapping from old (short) names to new (full) names using row order in summary files

import pandas as pd

# Load old (non-fixed) and new (fixed) summary files
batting_old = pd.read_excel('2024_raw_data.xlsx/batting_summary.xlsx')
batting_new = pd.read_csv('2024_raw_data.csv/batting_summary.csv')
bowling_old = pd.read_excel('2024_raw_data.xlsx/bowling_summary.xlsx')
bowling_new = pd.read_csv('2024_raw_data.csv/bowling_summary.csv')

# Map old to new names by row order for batters and bowlers
batting_name_map = dict(zip(batting_old['Player'], batting_new['Player']))
bowling_name_map = dict(zip(bowling_old['Player'], bowling_new['Player']))

# Merge both mappings (in case there are unique names in each)
name_map = {**batting_name_map, **bowling_name_map}

# Load deliveries file
deliveries = pd.read_csv('2024_raw_data.csv/season2024_deliveries.csv')

# Columns to fix
cols_to_fix = ['batter', 'bowler', 'non_striker', 'player_dismissed', 'fielder']

for col in cols_to_fix:
    if col in deliveries.columns:
        deliveries[col] = deliveries[col].map(name_map).fillna(deliveries[col])

# Save the fixed deliveries file
deliveries.to_csv('2024_raw_data.csv/season2024_deliveriesfixed.csv', index=False)

print("✅ All player name columns in season2024_deliveries.csv have been fixed using the new full names.")


In [ ]:
matches = pd.read_csv('2024_raw_data.csv/season2024_matches.csv')

matches['player_of_match'] = matches['player_of_match'].map(name_map).fillna(matches['player_of_match'])

# Save the fixed matches file
matches.to_csv('2024_raw_data.csv/season2024_matchesfixed.csv', index=False)

print("✅ Player of Match names in season2024_matches.csv have been fixed using the new full names.")